# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Baseline (Week 4):** rank pages by low h1 impressions. The fewer impressions, the higher the rank. Flag the top 10%.

**Models I try, simplest first:**
- **Logistic Regression** (log-scaled features). Simple, and the coefficients are easy to read. This is the first honest challenger.
- **Decision Tree** (shallow, depth 3). I can print the rules it learns, so I see what it uses.
- **Gradient Boosting** (HistGradientBoosting, shallow, regularised). It can catch non-linear signals. It is safe here because the data is small (about 17k rows) and I keep it shallow.

I do not pick a model because it is fancier. A model only counts as better if it beats the baseline on the same split with the same metric, by more than the fold-to-fold noise.

**Features:** only the five h1 features from my Week-3 contract: impressions_h1, clicks_h1, ctr_h1, avg_position_h1, active_days_h1. Nothing from 16-31 March goes in.

The code cell below loads the same data, filter and label as Week 4.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.execute(f"CREATE VIEW mar AS SELECT * FROM read_parquet('{MAR}')")

frame = con.sql("""
SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS avg_position_h1,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS active_days_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '2026-03-15') AS clicks_h2
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df = frame.copy()
df["clicks_h2"] = df["clicks_h2"].fillna(0)
df["ctr_h1"] = df["clicks_h1"] / df["impressions_h1"].replace(0, np.nan)
FEATURES = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
df = df[df["clicks_h1"] >= 5].dropna(subset=FEATURES).copy()
df["label"] = (df["clicks_h2"] < 0.8 * df["clicks_h1"]).astype(int)
df = df.reset_index(drop=True)

print("rows:", len(df), "| overall decline rate:", round(df["label"].mean(), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 17431 | overall decline rate: 0.442


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped 5-fold split by client.** All pages of one client stay together, either in train or in test, never both. My Week-4 check showed a few clients make up a big part of the data, so a random split would let the model learn "this client's pages" and look better than it is.

Every model and the baseline are scored on the **same out-of-fold rows**, so the comparison is fair. The baseline needs no training, so it is scored on those same rows.

Honest limit: this is one month, split inside the month (features from 1-15 March, label from 16-31 March). It is not time-aware across months and not a test on a different month.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold

GROUP = "client_hash_id"
y = df["label"].values
groups = df[GROUP].values

folds = list(GroupKFold(n_splits=5).split(df, y, groups))
for i, (tr, te) in enumerate(folds):
    assert set(groups[tr]).isdisjoint(groups[te]), "client leaked across train/test"
    print(f"fold {i}: train {len(tr):>6} | test {len(te):>6} | test decline rate {y[te].mean():.3f}")

print("\nclients:", df[GROUP].nunique(),
      "| biggest client share of rows:", round(df[GROUP].value_counts(normalize=True).iloc[0], 3))
assert not ({"label", "clicks_h2"} & set(FEATURES)), "label/future column in FEATURES"


fold 0: train  12821 | test   4610 | test decline rate 0.428
fold 1: train  14110 | test   3321 | test decline rate 0.478
fold 2: train  14264 | test   3167 | test decline rate 0.479
fold 3: train  14264 | test   3167 | test decline rate 0.403
fold 4: train  14265 | test   3166 | test decline rate 0.428

clients: 29 | biggest client share of rows: 0.264


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same folds, same rows, same metrics for everyone. Metrics:
- **ROC-AUC**: how well the score separates decliners from non-decliners (0.5 = coin flip).
- **PR-AUC**: precision-recall area. Compare it with the plain decline rate (the dumb guess).
- **Top-10% decline rate**: of the 10% of pages each method flags, how many really declined. This is the same number I used in Week 4 (baseline: 0.53 vs 0.442 for all rows).

**Result:**
- Baseline ROC-AUC 0.549 vs best model 0.586 (logistic regression). The gain is +0.037 pooled, and per fold it ranges from +0.014 to +0.058. The fold std is about 0.03 (0.026 for the baseline, 0.036 for logreg).
- Top-10% decline rate: baseline 0.530 vs logreg 0.562 (all rows 0.442). Lift goes from 1.20 to 1.27. PR-AUC goes from 0.484 to 0.513, against the dumb guess of 0.442.
- Logreg beats the baseline in 5 of 5 folds. HistGradientBoosting also wins 5 of 5 (ROC-AUC 0.577, top-10% 0.561). The decision tree wins only 3 of 5 (ROC-AUC 0.554).
- My verdict in plain words: the simplest model wins. Gradient boosting is not better than logistic regression, so the extra complexity bought nothing here. The gain over the baseline is small, about the size of the fold noise, but it is positive in every fold, so I call it a modest, directional improvement. In absolute terms every model is weak (ROC-AUC under 0.6), so this is decision-support for where to look, not a reliable predictor.

Directional only. One month, one split design.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import roc_auc_score, average_precision_score

X = df[FEATURES]

models = {
    "logreg": lambda: make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                                    LogisticRegression(max_iter=1000)),
    "tree_d3": lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42),
    "hist_gb": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                                      min_samples_leaf=100, l2_regularization=1.0,
                                                      random_state=42),
}

def oof_scores(make_model):
    s = np.zeros(len(X))
    for tr, te in folds:
        m = make_model().fit(X.iloc[tr], y[tr])
        s[te] = m.predict_proba(X.iloc[te])[:, 1]
    return s

scores = {"baseline (low impressions)": -df["impressions_h1"].values}
for name, mk in models.items():
    scores[name] = oof_scores(mk)

def top_rate(s, y_, frac=0.10):
    cut = np.quantile(s, 1 - frac)
    return y_[s >= cut].mean()

rows = []
for name, s in scores.items():
    fold_auc = [roc_auc_score(y[te], s[te]) for _, te in folds]
    rows.append({
        "model": name,
        "ROC-AUC": roc_auc_score(y, s),
        "ROC-AUC fold std": np.std(fold_auc),
        "PR-AUC": average_precision_score(y, s),
        "top-10% decline rate": top_rate(s, y),
        "lift vs all rows": top_rate(s, y) / y.mean(),
    })
results = pd.DataFrame(rows).set_index("model").round(3)
print("all-rows decline rate (dumb guess for PR-AUC):", round(y.mean(), 3))
display(results)

# fold by fold: does each model beat the baseline in every fold, or only on average?
base = scores["baseline (low impressions)"]
per_fold = pd.DataFrame({
    name: [roc_auc_score(y[te], s[te]) - roc_auc_score(y[te], base[te]) for _, te in folds]
    for name, s in scores.items() if name != "baseline (low impressions)"
}).round(3)
per_fold.index = [f"fold {i}" for i in range(len(folds))]
print("\nROC-AUC difference vs baseline, per fold (positive = model better):")
print(per_fold)
print("\nfolds won:", (per_fold > 0).sum().to_dict())


all-rows decline rate (dumb guess for PR-AUC): 0.442


,ROC-AUC,ROC-AUC fold std,PR-AUC,top-10% decline rate,lift vs all rows
model,,,,,
baseline (low impressions),0.549,0.026,0.484,0.530,1.199
logreg,0.586,0.036,0.513,0.562,1.271
tree_d3,0.554,0.024,0.483,0.514,1.163
hist_gb,0.577,0.031,0.504,0.561,1.269



ROC-AUC difference vs baseline, per fold (positive = model better):
        logreg  tree_d3  hist_gb
fold 0   0.058    0.024    0.058
fold 1   0.014    0.011    0.023
fold 2   0.022   -0.003    0.034
fold 3   0.029   -0.013    0.015
fold 4   0.029    0.015    0.019

folds won: {'logreg': 5, 'tree_d3': 3, 'hist_gb': 5}


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the errors look like:**

- Features the model leans on most: active_days_h1 and impressions_h1 (permutation importance, drop in AUC 0.055 and 0.049). The others are under 0.01, and ctr_h1 is about zero. This was measured on one fold (fold 0) only. Direction (logreg coefficients): more active days means more likely to decline (+0.33), more impressions means less likely (-0.20), more clicks slightly less likely (-0.09), worse position slightly more likely (+0.09), ctr about zero. I do not know why active days matters. It is a pattern in this month, not a cause.
- About 44% of flagged pages did not decline (decline rate among flagged is 0.562). The model still misses many decliners and flags many stable pages. Only about 54% of its flagged pages are also in the baseline's flagged set, so it picks different pages, not just the same ones.
- Errors by click band: the model flags mostly low-click pages. About 22% of pages with 5-7 clicks and 10% of pages with 8-10 clicks are flagged, but only 3% of 11-20 and 0.4% of 21+. About 73% of all flagged pages have only 5-7 clicks. The decline rate among flagged pages is about 0.52-0.57 in every band (the 21+ band has too few flagged pages to read). So the 5-7 click band is the noisiest: most flags are there, and one or two clicks can flip the label.
- Client concentration: 58% of flagged rows come from the top 3 clients (26 of 29 clients have at least one flagged row), so the result may reflect a few clients more than the whole panel.
- Likely reason it is hard: the label is built from clicks, and pages near the 5-click floor tend to fall back (regression to the mean). h1 features alone cannot tell content age, seasonality, or ranking changes.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance
from sklearn.tree import export_text

# what does the best model lean on? (held-out clients of fold 0)
best_name = results.drop(index="baseline (low impressions)")["ROC-AUC"].idxmax()
print("best model by OOF ROC-AUC:", best_name)

tr, te = folds[0]
m = models[best_name]().fit(X.iloc[tr], y[tr])
pi = permutation_importance(m, X.iloc[te], y[te], n_repeats=10, scoring="roc_auc", random_state=42)
imp = pd.DataFrame({"mean_drop_in_auc": pi.importances_mean, "std": pi.importances_std},
                   index=FEATURES).sort_values("mean_drop_in_auc", ascending=False).round(4)
print("\npermutation importance (bigger drop = model leans on it more):")
print(imp)

lr_all = models["logreg"]().fit(X, y)
coef = pd.Series(lr_all[-1].coef_[0], index=FEATURES).sort_values().round(3)
print("\nlogreg coefficients (standardised log features; + means more likely to decline):")
print(coef)

print("\ndecision tree rules:")
print(export_text(models["tree_d3"]().fit(X, y), feature_names=FEATURES))

# error analysis on out-of-fold scores of the best model (top 10% flagged)
d = df.copy()
d["score"] = scores[best_name]
d["flagged"] = d["score"] >= d["score"].quantile(0.90)
d["base_flagged"] = scores["baseline (low impressions)"] >= np.quantile(scores["baseline (low impressions)"], 0.90)

print("flagged:", int(d["flagged"].sum()),
      "| decline rate among flagged:", round(d.loc[d["flagged"], "label"].mean(), 3),
      "| share of flagged that did NOT decline:", round(1 - d.loc[d["flagged"], "label"].mean(), 3))
print("overlap with baseline's flagged pages:",
      round((d["flagged"] & d["base_flagged"]).sum() / d["flagged"].sum(), 3))

d["clicks_band"] = pd.cut(d["clicks_h1"], [4, 7, 10, 20, np.inf], labels=["5-7", "8-10", "11-20", "21+"])
by_band = d.groupby("clicks_band", observed=True).agg(
    n=("label", "size"),
    decline_rate=("label", "mean"),
    flagged_share=("flagged", "mean"),
).round(3)
by_band["decline_rate_when_flagged"] = d[d["flagged"]].groupby("clicks_band", observed=True)["label"].mean().round(3)
print("\nby h1 click band:")
print(by_band)

share = d.loc[d["flagged"], GROUP].value_counts(normalize=True)
print("\nflagged rows from top 3 clients:", round(share.head(3).sum(), 3), "| clients with any flagged row:", len(share))

pd.set_option("display.max_columns", 20)
display(d[d["flagged"] & (d["label"] == 0)].nlargest(10, "score")[
    ["score", "impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1", "clicks_h2"]].round(3))


best model by OOF ROC-AUC: logreg

permutation importance (bigger drop = model leans on it more):
                 mean_drop_in_auc     std
active_days_h1             0.0546  0.0043
impressions_h1             0.0493  0.0058
avg_position_h1            0.0091  0.0012
clicks_h1                  0.0075  0.0023
ctr_h1                    -0.0023  0.0010

logreg coefficients (standardised log features; + means more likely to decline):
impressions_h1    -0.195
clicks_h1         -0.094
ctr_h1             0.031
avg_position_h1    0.091
active_days_h1     0.334
dtype: float64

decision tree rules:
|--- active_days_h1 <= 13.50
|   |--- impressions_h1 <= 993.50
|   |   |--- active_days_h1 <= 10.50
|   |   |   |--- class: 0
|   |   |--- active_days_h1 >  10.50
|   |   |   |--- class: 1
|   |--- impressions_h1 >  993.50
|   |   |--- impressions_h1 <= 2068.50
|   |   |   |--- class: 0
|   |   |--- impressions_h1 >  2068.50
|   |   |   |--- class: 0
|--- active_days_h1 >  13.50
|   |--- clicks_h1 <= 24

,score,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,clicks_h2
6122,0.818,77.0,13.0,0.169,47.845,15,11.0
10384,0.754,96.0,9.0,0.094,20.998,15,10.0
8029,0.744,46.0,7.0,0.152,2.370,14,6.0
16863,0.726,103.0,13.0,0.126,4.325,15,14.0
13516,0.710,134.0,12.0,0.090,9.713,15,33.0
13851,0.700,126.0,5.0,0.040,12.862,15,5.0
2105,0.700,105.0,5.0,0.048,7.697,15,5.0
4647,0.696,167.0,5.0,0.030,21.916,15,10.0
13869,0.690,107.0,6.0,0.056,4.819,15,7.0
10380,0.684,268.0,16.0,0.060,30.530,15,23.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.